# Step-by-Step A2A (Agent-to-Agent) Experiment

This notebook walks you through building a multi-agent system using the **A2A protocol** — the same way you explored RAG and MCP step-by-step.

---

## What is A2A?

**A2A (Agent-to-Agent)** is an open protocol (created by Google, adopted by many AI vendors) that defines how AI agents communicate with **each other** over HTTP.

### The gap that A2A fills

You already know MCP: an LLM uses **tools** (vertical relationship — model → tools).
A2A solves a different problem: agents **delegating tasks** to other agents (horizontal relationship — agent ↔ agent).

```
  MCP (Model ↔ Tools)
  ─────────────────────────────────────────
  LLM  ──────────────►  calculator_server
        tool call               │
        ◄──────────────  42.0  ─┘

  A2A (Agent ↔ Agent)
  ─────────────────────────────────────────
  User
    │
    ▼
  Orchestrator Agent   ──► Math Agent
  (decides who to ask)  ◄── "The answer is 42"
                        ──► Writer Agent
                        ◄── "Here is your essay..."
```

### Key concepts

| Concept | What it is | Example |
|---|---|---|
| **Agent Card** | JSON descriptor advertising an agent's capabilities | `/.well-known/agent.json` |
| **Skill** | A specific capability an agent advertises | `"math"`, `"summarize"` |
| **Task** | The unit of work sent to an agent | A user question + conversation history |
| **Message** | A turn in the conversation (user or agent) | Text, file, or structured data |
| **Part** | A piece of a message | `TextPart`, `FilePart`, `DataPart` |
| **Artifact** | The agent's output | The final answer or generated file |

### A2A Transport

Unlike MCP (which defaults to stdio), A2A uses **HTTP + JSON-RPC 2.0**.
Agents are real HTTP services — discoverable, network-addressable, and composable.

```
  Client                         Agent Server
  ──────────────────────────────────────────────
  GET /.well-known/agent.json  ──►  agent card
  POST /  {method: "message/send", ...}  ──►  execute
                               ◄──  Task or Message
```

## Step 1 — Install dependencies

The Python SDK for A2A is `a2a-sdk`. We also need:
- `uvicorn`: ASGI server to run agent HTTP services
- `httpx`: async HTTP client for the A2A client side

> Note: `ollama` is already installed from `requirements.txt`.

In [1]:
%pip install a2a-sdk uvicorn httpx nest_asyncio2 --quiet

Note: you may need to restart the kernel to use updated packages.


## Step 2 — The Agent Card: how agents advertise themselves

An **Agent Card** is a JSON document served at `GET /.well-known/agent.json`.
It is the equivalent of an OpenAPI spec for agents — any A2A client can discover
what the agent does and how to talk to it.

Let's build one in Python to see its structure:

In [2]:
import json
from a2a.types import AgentCard, AgentCapabilities, AgentSkill

sample_card = AgentCard(
    name="Math Agent",
    description="Solves math problems step by step.",
    url="http://localhost:10002/",        # where the agent is reachable
    version="1.0.0",
    capabilities=AgentCapabilities(
        streaming=False,                 # does it support SSE streaming?
        pushNotifications=False,         # can it push task updates?
    ),
    skills=[
        AgentSkill(
            id="math-solver",
            name="Math Solver",
            description="Solves arithmetic, algebra, and basic calculus.",
            tags=["math", "calculation"],
        )
    ],
    defaultInputModes=["text/plain"],
    defaultOutputModes=["text/plain"],
)

# This is exactly what a client sees at /.well-known/agent.json
print(json.dumps(sample_card.model_dump(exclude_none=True), indent=2))

{
  "capabilities": {
    "pushNotifications": false,
    "streaming": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "description": "Solves math problems step by step.",
  "name": "Math Agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "Solves arithmetic, algebra, and basic calculus.",
      "id": "math-solver",
      "name": "Math Solver",
      "tags": [
        "math",
        "calculation"
      ]
    }
  ],
  "url": "http://localhost:10002/",
  "version": "1.0.0"
}


## Step 3 — Write your first A2A agent: the Echo Agent

An A2A agent server has three parts:

1. **`AgentExecutor`** — your business logic (subclass this)
2. **`DefaultRequestHandler`** — wires executor to the A2A protocol
3. **`A2AStarletteApplication`** — the ASGI HTTP app (served via uvicorn)

```
  HTTP Request
      │
      ▼
  A2AStarletteApplication   (routing, JSON-RPC parsing)
      │
      ▼
  DefaultRequestHandler     (task lifecycle management)
      │
      ▼
  AgentExecutor.execute()   ← YOUR CODE GOES HERE
      │
      ▼
  EventQueue.enqueue_event()   (sends response back)
```

Let's write a minimal echo agent:

In [3]:
echo_agent_code = '''
# echo_agent.py — the simplest possible A2A agent

import uvicorn
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.apps import A2AStarletteApplication
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCard, AgentCapabilities, AgentSkill
from a2a.utils import new_agent_text_message


class EchoAgentExecutor(AgentExecutor):
    """Receives any text and echoes it back with a prefix."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # context.get_user_input() extracts the text from the latest user message
        user_input = context.get_user_input()

        # enqueue_event is async in a2a-sdk >= 0.3.x — must be awaited
        await event_queue.enqueue_event(
            new_agent_text_message(f"Echo: {user_input}")
        )

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        # Called if a client cancels the task mid-flight
        pass


# --- Agent Card -----------------------------------------------------------
agent_card = AgentCard(
    name="Echo Agent",
    description="Echoes back whatever you send it.",
    url="http://localhost:10001/",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="echo",
            name="Echo",
            description="Echo the input message back.",
            tags=["echo", "test"]
        )
    ],
    defaultInputModes=["text/plain"],
    defaultOutputModes=["text/plain"],
)

# --- Wire everything together -------------------------------------------
request_handler = DefaultRequestHandler(
    agent_executor=EchoAgentExecutor(),
    task_store=InMemoryTaskStore(),  # stores task state in memory
)

app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=request_handler,
).build()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=10001, log_level="warning")
'''

with open("echo_agent.py", "w") as f:
    f.write(echo_agent_code.strip())

print("echo_agent.py written!")

echo_agent.py written!


## Step 4 — Start the agent and inspect its Agent Card

Unlike MCP (where the client *spawns* the server), A2A agents run as **independent HTTP services**.
We start the agent as a background subprocess and then connect to it over HTTP.

In [4]:
import subprocess
import time
import httpx
import asyncio
import nest_asyncio

nest_asyncio.apply()  # allow asyncio in Jupyter

# Start the echo agent as a background process
echo_proc = subprocess.Popen(
    ["python3", "echo_agent.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(2)  # give uvicorn a moment to start

# Verify it is running
if echo_proc.poll() is None:
    print("Echo agent is running (PID:", echo_proc.pid, ")")
else:
    stdout, stderr = echo_proc.communicate()
    print("Agent failed to start!")
    print(stderr.decode())

Echo agent is running (PID: 69184 )


In [5]:
import json

# Fetch the Agent Card — this is what any A2A client does first
# The well-known path is a convention from the A2A spec
async def fetch_agent_card(base_url: str):
    async with httpx.AsyncClient() as client:
        response = await client.get(f"{base_url}/.well-known/agent.json")
        return response.json()

card = asyncio.get_event_loop().run_until_complete(
    fetch_agent_card("http://localhost:10001")
)

print("=== Agent Card at /.well-known/agent.json ===")
print(json.dumps(card, indent=2))

=== Agent Card at /.well-known/agent.json ===
{
  "capabilities": {
    "streaming": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "description": "Echoes back whatever you send it.",
  "name": "Echo Agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "Echo the input message back.",
      "id": "echo",
      "name": "Echo",
      "tags": [
        "echo",
        "test"
      ]
    }
  ],
  "url": "http://localhost:10001/",
  "version": "1.0.0"
}


## Step 5 — Send your first A2A message

Now let's use `A2AClient` to send a message and receive a response.

The protocol uses **JSON-RPC 2.0** over HTTP POST to `/`.
The main method is `message/send`.

```
  POST http://localhost:10001/
  {
    "jsonrpc": "2.0",
    "id": "<request-id>",
    "method": "message/send",
    "params": {
      "message": {
        "role": "user",
        "parts": [{"kind": "text", "text": "hello!"}],
        "messageId": "<uuid>"
      }
    }
  }
```

In [6]:
# New API (a2a-sdk >= 0.3.x)
#
# What changed from 0.2.x:
#   OLD: A2AClient(httpx_client=..., agent_card=...)  → deprecated
#   NEW: ClientFactory.connect(url)                   → resolves card + creates client
#
#   OLD: client.send_message(SendMessageRequest(...)) → returns a response object
#   NEW: client.send_message(Message(...))            → async generator that yields events
#
# send_message now yields either:
#   - (Task, None)  — the completed task (Task-path agents)
#   - Message       — a direct message reply (Message-path agents)
#
# create_text_message_object(role, content='') — role is first, use content= keyword!
#
# TIMEOUT: default httpx timeout is 5s — too short for Ollama LLM calls (can take 4–30s).
# Always pass ClientConfig(httpx_client=httpx.AsyncClient(timeout=120.0)).

import httpx
from a2a.client import ClientFactory, ClientConfig
from a2a.client.helpers import create_text_message_object
from a2a.types import Task, Message


def extract_text(event) -> str:
    """Pull the agent's reply text from a send_message event."""
    if isinstance(event, tuple):
        # Task path: yields (Task, None)
        task, _ = event
        for msg in reversed(task.history or []):
            if msg.role.value == "agent":
                for part in msg.parts:
                    if hasattr(part.root, "text"):
                        return part.root.text
    elif isinstance(event, Message):
        # Message path: direct reply
        for part in event.parts:
            if hasattr(part.root, "text"):
                return part.root.text
    return str(event)


# Shared ClientConfig with a 60s timeout — enough for any local Ollama call.
# Re-used by call_agent and call_agent_raw so every notebook cell benefits.
_client_config = ClientConfig(httpx_client=httpx.AsyncClient(timeout=120.0))


async def call_agent(base_url: str, text: str) -> str:
    client = await ClientFactory.connect(base_url, client_config=_client_config)
    async for event in client.send_message(create_text_message_object(content=text)):
        return extract_text(event)
    return ""


reply = asyncio.get_event_loop().run_until_complete(
    call_agent("http://localhost:10001", "Hello, world!")
)
print(f"Agent replied: {reply}")

Agent replied: Echo: Hello, world!


In [7]:
# Inspect the raw event to understand what send_message yields
async def call_agent_raw(base_url: str, text: str):
    client = await ClientFactory.connect(base_url, client_config=_client_config)
    async for event in client.send_message(create_text_message_object(content=text)):
        return event  # return the raw Python object

raw = asyncio.get_event_loop().run_until_complete(
    call_agent_raw("http://localhost:10001", "Show me the protocol!")
)

print(f"Type  : {type(raw)}")
print()

if isinstance(raw, tuple):
    task, _ = raw
    print(f"Task id     : {task.id}")
    print(f"Task status : {task.status.state}")
    print(f"History     :")
    for msg in task.history or []:
        print(f"  [{msg.role.value}] ", end="")
        for part in msg.parts:
            if hasattr(part.root, "text"):
                print(part.root.text[:120])
elif isinstance(raw, Message):
    print(f"Message role : {raw.role.value}")
    for part in raw.parts:
        if hasattr(part.root, "text"):
            print(part.root.text)

Type  : <class 'a2a.types.Message'>

Message role : agent
Echo: Show me the protocol!


## Step 6 — What just happened? Message vs Task responses

The raw event you inspected was a `Message`, not a `(Task, None)` tuple. This is because `new_agent_text_message()` sets `kind='message'`, which the SDK short-circuits directly — the Task machinery is never invoked.

```
AgentExecutor enqueues...       ResultAggregator returns...   Client yields...
─────────────────────────────   ───────────────────────────   ────────────────
new_agent_text_message(...)  →  Message                    →  Message
TaskStatusUpdateEvent(...)   →  Task (after final=True)    →  (Task, None)
TaskArtifactUpdateEvent(...) →  Task (after final=True)    →  (Task, None)
```

### The full Task structure (for the Task path)

```
  (Task, None)
   └── Task
       ├── id          — unique task identifier (UUID)
       ├── status
       │   └── state   — "completed" | "working" | "failed" | "canceled"
       ├── history     — the conversation turns (user + agent messages)
       └── artifacts   — structured outputs (files, data blobs)
```

### Task states

```
  submitted ──► working ──► completed
                   │
                   ├──► failed
                   └──► canceled
```

**When to use each path:**

| Use `new_agent_text_message` | Use Task events |
|---|---|
| Simple text reply, single turn | Structured/multi-part output |
| No state tracking needed | Long-running work (working → completed) |
| Matches the echo/chat pattern | Artifacts (files, JSON data) |

In [8]:
# Stop the echo agent — we are done with it
echo_proc.terminate()
echo_proc.wait()
print("Echo agent stopped.")

Echo agent stopped.


## Step 7 — Force the Task path: the Stats Agent

To see a real `(Task, None)` response we need an agent that enqueues **Task-typed events** instead of a plain `Message`.

The Stats Agent takes any sentence and returns word/character counts as a structured `DataPart` artifact. It explicitly signals:

1. `TaskStatusUpdateEvent(state=working, final=False)` — "I started"
2. `TaskArtifactUpdateEvent(...)` — "here is the structured result"
3. `TaskStatusUpdateEvent(state=completed, final=True)` — "I am done"

```
Client                            Stats Agent (server)
──────                            ────────────────────
send_message("hello world")  →
                                  enqueue TaskStatusUpdateEvent(working)
                                  enqueue TaskArtifactUpdateEvent({words:2, chars:11})
                                  enqueue TaskStatusUpdateEvent(completed, final=True)
                             ←   returns Task object with .artifacts populated
(Task, None) ←────────────────
task.artifacts[0].parts[0]        ← DataPart(data={"words": 2, "chars": 11})
```

In [9]:
stats_agent_code = '''
# stats_agent.py — demonstrates the Task response path
#
# Unlike the Echo Agent (which enqueues a Message via new_agent_text_message),
# this agent enqueues Task-typed events, which forces the SDK to return a
# (Task, None) tuple to the client instead of a plain Message.

import uvicorn
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.apps import A2AStarletteApplication
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import (
    AgentCard, AgentCapabilities, AgentSkill,
    TaskState, TaskStatus,
    TaskStatusUpdateEvent,   # signals state changes (working, completed, …)
    TaskArtifactUpdateEvent, # attaches structured output to the Task
    Artifact, Part, DataPart,
)


class StatsAgentExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        text    = context.get_user_input()
        task_id = context.task_id
        ctx_id  = context.context_id

        # Step 1 — Signal that work has started.
        # This creates the Task object in the server\'s TaskStore.
        await event_queue.enqueue_event(TaskStatusUpdateEvent(
            task_id=task_id,
            context_id=ctx_id,
            status=TaskStatus(state=TaskState.working),
            final=False,   # more events will follow
        ))

        # Step 2 — Compute the result and attach it as a structured artifact.
        # DataPart holds arbitrary JSON-serialisable data (dict, list, etc.).
        stats = {
            "words":           len(text.split()),
            "chars":           len(text),
            "chars_no_spaces": len(text.replace(" ", "")),
        }
        await event_queue.enqueue_event(TaskArtifactUpdateEvent(
            task_id=task_id,
            context_id=ctx_id,
            artifact=Artifact(
                artifact_id="stats",
                name="Text statistics",
                parts=[Part(DataPart(data=stats))],
            ),
        ))

        # Step 3 — Signal completion. final=True tells the aggregator to
        # stop consuming and return the Task to the client.
        await event_queue.enqueue_event(TaskStatusUpdateEvent(
            task_id=task_id,
            context_id=ctx_id,
            status=TaskStatus(state=TaskState.completed),
            final=True,
        ))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass


agent_card = AgentCard(
    name="Stats Agent",
    description="Returns word and character counts for any input text as a structured artifact.",
    url="http://localhost:10005/",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="stats",
            name="Text Statistics",
            description="Count words and characters in a string.",
            tags=["stats", "text"],
        )
    ],
    defaultInputModes=["text/plain"],
    defaultOutputModes=["application/json"],
)

request_handler = DefaultRequestHandler(
    agent_executor=StatsAgentExecutor(),
    task_store=InMemoryTaskStore(),
)

app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=request_handler,
).build()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=10005, log_level="warning")
'''

with open("stats_agent.py", "w") as f:
    f.write(stats_agent_code.strip())

print("stats_agent.py written!")

stats_agent.py written!


In [10]:
# Start the Stats Agent as a subprocess on port 10005
stats_proc = subprocess.Popen(
    ["python3", "stats_agent.py"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
)
time.sleep(1.5)

if stats_proc.poll() is not None:
    print("Stats agent failed to start!")
    print(stats_proc.stderr.read().decode())
else:
    print("Stats agent running on http://localhost:10005")

Stats agent running on http://localhost:10005


In [11]:
# Send a message and inspect the RAW event — this time it should be (Task, None)
raw = asyncio.get_event_loop().run_until_complete(
    call_agent_raw("http://localhost:10005", "the quick brown fox")
)

print(f"Event type : {type(raw)}")
print()

if isinstance(raw, tuple):
    task, _ = raw
    print(f"✓ Got a Task!")
    print(f"  task.id            : {task.id}")
    print(f"  task.status.state  : {task.status.state}")
    print(f"  task.artifacts     : {len(task.artifacts or [])} artifact(s)")
    print()

    # Extract the DataPart from the first artifact
    artifact = task.artifacts[0]
    print(f"  artifact.name      : {artifact.name}")
    data = artifact.parts[0].root.data   # DataPart.data holds the dict
    print(f"  artifact data      : {data}")

elif isinstance(raw, Message):
    print("Got a Message — check that the agent enqueues Task events, not new_agent_text_message()")

Event type : <class 'tuple'>

✓ Got a Task!
  task.id            : b670e269-4fd0-4049-8c3b-63c04966a026
  task.status.state  : TaskState.completed
  task.artifacts     : 1 artifact(s)

  artifact.name      : Text statistics
  artifact data      : {'words': 4, 'chars': 19, 'chars_no_spaces': 16}


In [12]:
# Try a few more inputs to see the stats vary
for sentence in [
    "hello",
    "A2A makes multi-agent systems composable and interoperable",
    "one two three four five",
]:
    raw = asyncio.get_event_loop().run_until_complete(
        call_agent_raw("http://localhost:10005", sentence)
    )
    task, _ = raw
    data = task.artifacts[0].parts[0].root.data
    print(f"Input  : {sentence!r}")
    print(f"Output : words={data['words']}  chars={data['chars']}  chars_no_spaces={data['chars_no_spaces']}")
    print()

Input  : 'hello'
Output : words=1  chars=5  chars_no_spaces=5

Input  : 'A2A makes multi-agent systems composable and interoperable'
Output : words=7  chars=58  chars_no_spaces=52

Input  : 'one two three four five'
Output : words=5  chars=23  chars_no_spaces=19



In [13]:
# Stop the Stats Agent
stats_proc.terminate()
stats_proc.wait()
print("Stats agent stopped.")

Stats agent stopped.


## Step 8 — Build a real agent backed by Ollama: the Math Agent

Now let's make something genuinely useful. The **Math Agent** receives a maths question, passes it to `llama3.2` via Ollama, and returns the answer — all over A2A.

```
Client ──► Math Agent ──► Ollama (llama3.2) ──► answer ──► Client
```

> Make sure Ollama is running (`ollama serve`) before starting this agent.

In [14]:
math_agent_code = '''
# math_agent.py — an A2A agent that solves math using llama3.2

import uvicorn
import ollama
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.apps import A2AStarletteApplication
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCard, AgentCapabilities, AgentSkill
from a2a.utils import new_agent_text_message

SYSTEM_PROMPT = """You are a math expert. Solve math problems step by step.
Show your reasoning clearly. Be concise."""


class MathAgentExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        user_input = context.get_user_input()

        # Call the local LLM via Ollama
        response = ollama.chat(
            model="llama3.2",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_input},
            ],
        )
        answer = response["message"]["content"]
        # enqueue_event is async in a2a-sdk >= 0.3.x — must be awaited
        await event_queue.enqueue_event(new_agent_text_message(answer))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass


agent_card = AgentCard(
    name="Math Agent",
    description="Solves math problems using llama3.2 running via Ollama.",
    url="http://localhost:10002/",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="math-solver",
            name="Math Solver",
            description="Solve arithmetic, algebra, and basic calculus problems.",
            tags=["math", "calculation", "algebra"],
        )
    ],
    defaultInputModes=["text/plain"],
    defaultOutputModes=["text/plain"],
)

request_handler = DefaultRequestHandler(
    agent_executor=MathAgentExecutor(),
    task_store=InMemoryTaskStore(),
)

app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=request_handler,
).build()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=10002, log_level="warning")
'''

with open("math_agent.py", "w") as f:
    f.write(math_agent_code.strip())

print("math_agent.py written!")

math_agent.py written!


In [15]:
# Start the math agent
math_proc = subprocess.Popen(
    ["python3", "math_agent.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(2)

if math_proc.poll() is None:
    print("Math agent running (PID:", math_proc.pid, ")")
else:
    stdout, stderr = math_proc.communicate()
    print("Agent failed to start:", stderr.decode())

Math agent running (PID: 69209 )


In [16]:
# Ask the Math Agent a question
answer = asyncio.get_event_loop().run_until_complete(
    call_agent("http://localhost:10002", "What is 17 * 23 + the square root of 144?")
)
print(answer)

To solve this problem, we need to follow the order of operations (PEMDAS):

1. Multiply 17 and 23:
17 × 23 = 391
2. Find the square root of 144:
√144 = √(12²) = 12

Now, add the results together:
391 + 12 = 403


In [17]:
# Try another question
answer = asyncio.get_event_loop().run_until_complete(
    call_agent("http://localhost:10002", "If I have 3x + 7 = 22, what is x?")
)
print(answer)

To solve for x, we need to isolate the variable x on one side of the equation.

Given equation: 3x + 7 = 22

Subtract 7 from both sides:

3x + 7 - 7 = 22 - 7
3x = 15

Now, divide both sides by 3:

3x / 3 = 15 / 3
x = 5


## Step 9 — The Orchestrator Pattern: A2A's killer feature

Now we unlock what makes A2A powerful: **agents calling other agents**.

The **Orchestrator Agent** uses an LLM to classify the incoming request, then routes it to the right specialist — all using the standard A2A protocol.

```
User
 │
 ▼
Orchestrator Agent (port 10004)
 │   classifies request with llama3.2
 ├──► "math"    ──► Math Agent   (port 10002)
 └──► "writing" ──► Writer Agent (port 10003)
```

The orchestrator is itself an A2A agent — any caller treats it exactly like any other agent, unaware of the delegation happening underneath.

In [18]:
writer_agent_code = '''
# writer_agent.py — A2A agent for writing and summarization tasks

import uvicorn
import ollama
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.apps import A2AStarletteApplication
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCard, AgentCapabilities, AgentSkill
from a2a.utils import new_agent_text_message

SYSTEM_PROMPT = """You are a skilled writer and editor.
Help with drafting emails, stories, and essays. Improve tone and clarity.
Summarize long texts into clear, concise bullet points.
Keep responses focused and polished."""


class WriterAgentExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        user_input = context.get_user_input()
        response = ollama.chat(
            model="llama3.2",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_input},
            ],
        )
        answer = response["message"]["content"]
        # enqueue_event is async in a2a-sdk >= 0.3.x — must be awaited
        await event_queue.enqueue_event(new_agent_text_message(answer))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass


agent_card = AgentCard(
    name="Writer Agent",
    description="Drafts, edits, and summarizes text using llama3.2.",
    url="http://localhost:10003/",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="write",
            name="Writing Assistant",
            description="Draft, edit, and improve written content.",
            tags=["writing", "editing", "email"],
        ),
        AgentSkill(
            id="summarize",
            name="Summarizer",
            description="Summarize long text into concise bullet points.",
            tags=["summarization", "tldr"],
        ),
    ],
    defaultInputModes=["text/plain"],
    defaultOutputModes=["text/plain"],
)

request_handler = DefaultRequestHandler(
    agent_executor=WriterAgentExecutor(),
    task_store=InMemoryTaskStore(),
)

app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=request_handler,
).build()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=10003, log_level="warning")
'''

with open("writer_agent.py", "w") as f:
    f.write(writer_agent_code.strip())

print("writer_agent.py written!")

writer_agent.py written!


In [19]:
orchestrator_code = '''
# orchestrator_agent.py
# An A2A agent that routes requests to specialist agents via A2A protocol.

import httpx
import uvicorn
import ollama
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.apps import A2AStarletteApplication
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCard, AgentCapabilities, AgentSkill, Task, Message
from a2a.client import ClientFactory, ClientConfig
from a2a.client.helpers import create_text_message_object
from a2a.utils import new_agent_text_message

MATH_AGENT_URL   = "http://localhost:10002"
WRITER_AGENT_URL = "http://localhost:10003"

# 60s timeout — Ollama calls on the specialist agents can take 4–30s
_client_config = ClientConfig(httpx_client=httpx.AsyncClient(timeout=120.0))

ROUTER_PROMPT = """You are a routing assistant. Classify the request as exactly one word:
- "math"    if it involves numbers, calculations, equations, or formulas
- "writing" if it involves text, drafting, editing, or summarization
Reply with ONLY the single word."""


def extract_text(event) -> str:
    if isinstance(event, tuple):
        task, _ = event
        for msg in reversed(task.history or []):
            if msg.role.value == "agent":
                for part in msg.parts:
                    if hasattr(part.root, "text"):
                        return part.root.text
    elif isinstance(event, Message):
        for part in event.parts:
            if hasattr(part.root, "text"):
                return part.root.text
    return str(event)


class OrchestratorExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        user_input = context.get_user_input()

        # ── Step 1: Classify the request with LLM ─────────────────────────────
        classification = ollama.chat(
            model="llama3.2",
            messages=[
                {"role": "system", "content": ROUTER_PROMPT},
                {"role": "user",   "content": user_input},
            ],
        )["message"]["content"].strip().lower()

        # ── Step 2: Route to the right specialist ─────────────────────────────
        if "math" in classification:
            target_url  = MATH_AGENT_URL
            target_name = "Math Agent"
        else:
            target_url  = WRITER_AGENT_URL
            target_name = "Writer Agent"

        # ── Step 3: Call the specialist agent via A2A ─────────────────────────
        specialist_client = await ClientFactory.connect(target_url, client_config=_client_config)
        specialist_text = ""
        async for event in specialist_client.send_message(
            create_text_message_object(content=user_input)
        ):
            specialist_text = extract_text(event)
            break

        # ── Step 4: Return the specialist\'s answer to the original caller ────
        reply = f"[Routed to {target_name}]\\n\\n{specialist_text}"
        await event_queue.enqueue_event(new_agent_text_message(reply))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass


agent_card = AgentCard(
    name="Orchestrator Agent",
    description="Routes user requests to Math or Writer specialist agents.",
    url="http://localhost:10004/",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="route",
            name="Smart Router",
            description="Automatically delegates tasks to the right specialist agent.",
            tags=["orchestration", "routing"],
        )
    ],
    defaultInputModes=["text/plain"],
    defaultOutputModes=["text/plain"],
)

request_handler = DefaultRequestHandler(
    agent_executor=OrchestratorExecutor(),
    task_store=InMemoryTaskStore(),
)

app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=request_handler,
).build()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=10004, log_level="warning")
'''

with open("orchestrator_agent.py", "w") as f:
    f.write(orchestrator_code.strip())

print("orchestrator_agent.py written!")

orchestrator_agent.py written!


## Step 10 — Start all agents and test the full pipeline

In [20]:
# Start Writer Agent and Orchestrator Agent
# (Math Agent is already running from Step 7)

writer_proc = subprocess.Popen(
    ["python3", "writer_agent.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
orchestrator_proc = subprocess.Popen(
    ["python3", "orchestrator_agent.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(2)  # wait for both to boot

all_procs = {
    "Math Agent      (10002)": math_proc,
    "Writer Agent    (10003)": writer_proc,
    "Orchestrator    (10004)": orchestrator_proc,
}
for name, proc in all_procs.items():
    status = "running" if proc.poll() is None else "FAILED"
    print(f"  {name}  →  {status}")

  Math Agent      (10002)  →  running
  Writer Agent    (10003)  →  running
  Orchestrator    (10004)  →  running


In [21]:
# Test 1: A math question — should be routed to Math Agent
print("=== Math question ===")
answer = asyncio.get_event_loop().run_until_complete(
    call_agent("http://localhost:10004", "What is 256 divided by 16, minus 4 squared?")
)
print(answer)

=== Math question ===
[Routed to Math Agent]

To solve this problem, I'll follow the order of operations (PEMDAS):

1. Divide 256 by 16:
   256 ÷ 16 = 16

2. Square 4:
   4² = 4 × 4 = 16

3. Subtract 16 from 16:
   16 - 16 = 0


In [22]:
# Test 2: A writing question — should be routed to Writer Agent
print("=== Writing question ===")
answer = asyncio.get_event_loop().run_until_complete(
    call_agent(
        "http://localhost:10004",
        "Write a short professional email declining a meeting invitation politely."
    )
)
print(answer)

=== Writing question ===
[Routed to Writer Agent]

Here is a sample email:

Subject: Regrettable Decline of Meeting Invitation

Dear [Name],

I appreciate the invitation to meet with you on [Date] at [Location]. I am grateful for the opportunity to discuss [Topic].

Unfortunately, my current schedule does not allow me to accommodate this meeting. I regret any inconvenience this may cause and would be happy to reschedule for a later date if needed.

Please let me know if there is an alternative time that works for you, or if there are any other ways we can stay in touch.

Thank you again for considering me, and I look forward to the possibility of connecting with you in the future.

Best regards,
[Your Name]


In [23]:
# Test 3: Summarization — should be routed to Writer Agent
print("=== Summarization question ===")
answer = asyncio.get_event_loop().run_until_complete(
    call_agent(
        "http://localhost:10004",
        "Summarize what A2A protocol is in 3 bullet points."
    )
)
print(answer)

=== Summarization question ===
[Routed to Writer Agent]

Here are three bullet points summarizing the A2A (Asynchronous Application-to-Application) protocol:

• **Definition**: The A2A protocol is a messaging system that enables applications to communicate with each other asynchronously, allowing them to send and receive data independently without requiring real-time responses.

• **Key Features**: A2A protocols are designed to facilitate the exchange of messages between multiple applications, allowing for decoupling and scalability in software architecture. They often use a push-pull model, where one application pushes updates to another, or both parties pull updates from each other.

• **Advantages**: The A2A protocol offers several benefits, including improved fault tolerance, enhanced scalability, and reduced latency compared to traditional synchronous communication models. This makes it suitable for real-time applications such as live streaming, gaming, and IoT data processing.


In [24]:
# Clean up — stop all agent processes
for name, proc in all_procs.items():
    proc.terminate()
    proc.wait()
    print(f"Stopped {name.strip()}")

Stopped Math Agent      (10002)
Stopped Writer Agent    (10003)
Stopped Orchestrator    (10004)


## Step 11 — A2A vs MCP: when to use which

| | MCP | A2A |
|---|---|---|
| **Transport** | stdio (or HTTP) | HTTP (always) |
| **Discovery** | Server spawned as subprocess | Agent Card at `/.well-known/agent.json` |
| **Unit of work** | Tool call (function + args) | Task (message + history) |
| **Who calls it** | An LLM client (e.g. Claude) | Another agent or a user app |
| **State** | Stateless per call | Stateful Task with history |
| **Best for** | Giving an LLM access to tools | Agent-to-agent delegation |

### Response type summary

| Executor enqueues | Client receives | Use when |
|---|---|---|
| `new_agent_text_message(...)` | `Message` | Simple text reply, single turn |
| `TaskStatusUpdateEvent` + `TaskArtifactUpdateEvent` | `(Task, None)` | Structured output, long-running work |

### What you built

| Step | What you did |
|---|---|
| 1 | Installed `a2a-sdk`, `uvicorn`, `httpx` |
| 2 | Understood the Agent Card |
| 3 | Wrote the Echo Agent (Message path) |
| 4 | Started it and fetched its Agent Card over HTTP |
| 5 | Sent your first A2A message with `ClientFactory` |
| 6 | Understood Message vs Task response paths |
| 7 | Wrote the Stats Agent (Task path with artifacts) |
| 8 | Added LLM intelligence via Ollama (Math Agent) |
| 9 | Built the Orchestrator pattern (agent-to-agent routing) |
| 10 | Tested the full multi-agent pipeline |

### Where to go next

- **Push notifications**: configure a webhook on `MessageSendConfiguration` so the server calls you back instead of blocking
- **Streaming**: set `capabilities=AgentCapabilities(streaming=True)` and use `send_message_streaming` for token-by-token output
- **Multi-turn conversations**: reuse `context_id` across calls to maintain conversation history in the TaskStore
- **A2A spec**: https://google.github.io/A2A